# Rarefied District Richness Comparison

Equal-cell rarefaction to compare districts fairly under uneven sampling effort.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='paper')
RANDOM_SEED = 42
N_ITER = 300


In [ ]:
p = next((x for x in [Path('file6.csv'), Path('../file6.csv'), Path('../../file6.csv')] if x.exists()), None)
if p is None:
    raise FileNotFoundError('file6.csv not found')
cols = ['stateProvince','verbatimScientificName','decimalLatitude','decimalLongitude','eventDate']
df = pd.read_csv(p, usecols=cols, low_memory=False).dropna(subset=cols)
for c in ['decimalLatitude','decimalLongitude']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=['decimalLatitude','decimalLongitude'])


In [ ]:
lon0 = float(df['decimalLongitude'].median())
lat0 = float(df['decimalLatitude'].median())
lat_rad = np.radians(df['decimalLatitude'].to_numpy())
df['x_km'] = (df['decimalLongitude'].to_numpy() - lon0) * 111.320 * np.cos(lat_rad)
df['y_km'] = (df['decimalLatitude'].to_numpy() - lat0) * 110.574
df['grid_x'] = np.floor(df['x_km'] / 5.0).astype(int)
df['grid_y'] = np.floor(df['y_km'] / 5.0).astype(int)
df['cell_id'] = df['stateProvince'].astype(str) + '|' + df['grid_x'].astype(str) + '_' + df['grid_y'].astype(str)
df['_key'] = df['cell_id'] + '|' + df['verbatimScientificName']
rng = np.random.default_rng(RANDOM_SEED)
df['_r'] = rng.random(len(df))
df_thin = df.sort_values('_r').groupby('_key', as_index=False).head(1)
cell_summary = df_thin.groupby(['stateProvince','cell_id'], as_index=False).agg(species_richness=('verbatimScientificName','nunique'))


In [ ]:
cells_per = cell_summary.groupby('stateProvince').size().sort_values()
min_cells = int(cells_per.min())
print('Common sampled cells per district:', min_cells)
by_d = {d:s.reset_index(drop=True) for d,s in cell_summary.groupby('stateProvince')}
rng = np.random.default_rng(RANDOM_SEED)
rows=[]
for i in range(N_ITER):
    for d,s in by_d.items():
        idx = rng.choice(len(s), size=min_cells, replace=False)
        sample = s.iloc[idx]
        rows.append({'iter':i,'stateProvince':d,'mean_cell_richness':sample['species_richness'].mean()})
rare = pd.DataFrame(rows)
summary = rare.groupby('stateProvince', as_index=False).agg(median_richness=('mean_cell_richness','median'),q025=('mean_cell_richness',lambda x: np.quantile(x,0.025)),q975=('mean_cell_richness',lambda x: np.quantile(x,0.975))).sort_values('median_richness', ascending=False)
summary.head(30)


In [ ]:
s = summary.sort_values('median_richness', ascending=True)
x = s['median_richness'].to_numpy()
xerr = np.vstack([x - s['q025'].to_numpy(), s['q975'].to_numpy()-x])
plt.figure(figsize=(10, max(5.5,0.3*len(s))))
plt.errorbar(x, s['stateProvince'], xerr=xerr, fmt='o', color='#1f618d', ecolor='#7fb3d5', capsize=3)
plt.title('Rarefied district mean cell richness (median and 95% interval)')
plt.xlabel('Mean cell richness')
plt.tight_layout()
plt.show()
